# Повторение RAG-эксперимента

Выберите GPU T4 и добавьте `GROQ_API_KEY` в Colab Secrets с доступом для ноутбука. Выполняйте ячейки сверху вниз. Данные уже в репозитории. Эта версия запускает новый эксперимент; исходный выполненный прогон 25.09.2026 сохранён в `01_colab.ipynb`. Новые ответы и оценки могут отличаться из-за внешнего API.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
root = Path('/content/rag-homework-reproduce')
if not root.exists():
    subprocess.run(['git', 'clone', '--branch', 'rag-experiments',
                    'https://github.com/evelinashakhnazaryan/rag-homework.git', str(root)], check=True)
os.chdir(root)
print('Project:', root.name)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
installation = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'],
                              capture_output=True, text=True)
print(installation.stdout[-4000:] + installation.stderr[-4000:])
assert installation.returncode == 0

In [ ]:
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
config = json.loads(Path('configs/default.json').read_text(encoding='utf-8'))
config['results_dir'] = 'results_reproduction'
config_path = Path('configs/reproduction.json')
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
def run_stage(mode):
    command = [sys.executable, '-u', '-m', 'src.run', '--config', str(config_path), '--mode', mode]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    assert status == 0, f'{mode}: ошибка, повторите ячейку для продолжения с checkpoint'
run_stage('validate')
print(json.dumps(config, ensure_ascii=False, indent=2))

## API без RAG

In [ ]:
run_stage('groq_zero')

## Локальная Qwen через vLLM

In [ ]:
run_stage('local_zero')

## Та же API-модель с LangChain RAG

In [ ]:
run_stage('groq_rag')

## Итоговый отчёт

In [ ]:
result = subprocess.run([sys.executable, '-m', 'src.report', '--config', str(config_path)],
                        capture_output=True, text=True)
assert result.returncode == 0, result.stderr
from IPython.display import Markdown, display
display(Markdown(Path(config['results_dir'], 'REPORT.md').read_text(encoding='utf-8')))
import importlib.metadata, platform
env = {'python': platform.python_version(),
       'packages': {d.metadata['Name']: d.version for d in importlib.metadata.distributions()}}
Path(config['results_dir'], 'environment_final.json').write_text(json.dumps(env, indent=2), encoding='utf-8')

## Сохранение

Скачайте `.ipynb` через меню Файл → Скачать: выводы сохранятся вместе с кодом. Следующая ячейка скачивает все результаты. Для нового независимого прогона смените `results_dir`; повторный запуск с тем же каталогом продолжает уже сохранённые ответы.

In [ ]:
import zipfile
from google.colab import files
with zipfile.ZipFile('/content/rag-reproduction-results.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for p in Path(config['results_dir']).glob('*'):
        if p.is_file():
            archive.write(p, str(p))
files.download('/content/rag-reproduction-results.zip')